In [1]:
print("hello")

hello


In [1]:
import requests
from models import Attraction, Venue, Event
from typing import Dict, Any, List, Tuple

In [2]:
tixmaster_key = "xvEJ5jaX7DcWdGqDCQX4HlXWTdby0nkb"

base_url = "https://app.ticketmaster.com/discovery/v2/events"
# base_url = "https://app.ticketmaster.com/discovery/v2/events/rZ7HnEZ1AfPU-N"

In [3]:
max_pages = 1
page_size = 2
start_date = "2026-05-01T00:00:00Z"

def get_data():
    # url = f"{base_url}?apikey={tixmaster_key}&locale=*&size=1"
    url = f"{base_url}?apikey={tixmaster_key}&locale=*&startDateTime={start_date}&size={page_size}&sort=date,asc"
    response = requests.get(url)

    if response.status_code == 200:
        events_json = response.json()
        return events_json
    else:
        print(f"Failed to retrieve data {response.status_code}")

In [4]:
event_info = get_data()

In [5]:
event_info

{'_embedded': {'events': [{'name': 'Vrienden van Bluesroute Helmond 2026',
    'type': 'event',
    'id': 'LvZ18QLuYI90z8YvqPd99',
    'test': False,
    'description': 'Word jij Vriend van de 15e Bluesroute Helmond? 🎸\xa0\nHelp ons het grootste gratis regionale bluesfestival in leven te houden!\nIn 2026 vieren we een bijzonder jubileum: de 15e editie van Bluesroute Helmond! Al vijftien jaar brengen we de ziel van de blues naar het hart van onze stad. Om dit jubileumjaar onvergetelijk te maken én het festival ook in de toekomst gratis toegankelijk te houden, hebben we jouw steun harder nodig dan ooit.\nVier 15 jaar passie en muziek\nAls Vriend steun je niet alleen de organisatie en de vrijwilligers, maar ook de artiesten die Helmond drie dagen lang trakteren op een ongekende dosis blues van wereldniveau. Dankzij jouw bijdrage profiteren lokale brouwers, horeca en winkeliers mee, en blijft Helmond dé blues-hoofdstad van Nederland.\nZie jouw vriendschap als een symbolisch ticket voor dit

In [12]:
def transform_data(raw_json: Dict[str, Any]) -> Tuple[List[Event], List[Venue], List[Attraction]]:
    events: List[Event] = []
    venues_dict: Dict[str, Venue] = {}
    attractions_dict: Dict[str, Attraction] = {}

    raw_events = raw_json.get("_embedded", {}).get("events", [])

    for item in raw_events:
        # --- 1. Parse Venue ---
        venue_id = None
        venue_item = item.get("_embedded", {}).get("venues", [])
        if venue_item:
            v = venue_item[0]
            venue_id = v.get("id")

            # only save if new venue_id
            if venue_id and venue_id not in venues_dict:
                venues_dict[venue_id] = Venue(
                    venue_id=venue_id,
                    name=v.get("name"),
                    city=v.get("city", {}).get("name"),
                    state=v.get("state", {}).get("stateCode"),
                    country=v.get("country", {}).get("countryCode"),
                    postal_code=v.get("postalCode"),
                    longitude=v.get("location", {}).get("longitude"),
                    latitude=v.get("location", {}).get("latitude")
                )

        # --- 2. Parse Attraction ---
        attraction_id = None
        attraction_item = item.get("_embedded", {}).get("attractions", [])
        if attraction_item:
            a = attraction_item[0]
            attraction_id = a.get("id")

            # only save if new att_id
            if attraction_id and attraction_id not in attractions_dict:
                att_classifications = a.get("classifications", [{}])[0]
                attractions_dict[attraction_id] = Attraction(
                    attraction_id=attraction_id,
                    # attraction_id=attraction_id,
                    name=a.get("name"),
                    segment=att_classifications.get("segment").get("name"),
                    genre=att_classifications.get("genre").get("name"),
                    subgenre=att_classifications.get("subGenre").get("name")
                )

        # --- 3. Parse Event ---
        price_ranges = item.get("priceRanges", [{}])[0]
        classifications = item.get("classifications", [{}])[0]
        events.append(Event(
            event_id=item.get("id"),
            name=item.get("name"),
            date=item.get("dates").get("start").get("dateTime"),
            venue_id=venue_id,
            attraction_id=attraction_id,
            min_price=price_ranges.get("min"),
            max_price=price_ranges.get("max"),
            segment=classifications.get("segment").get("name"),
            genre=classifications.get("genre").get("name"),
            subgenre=classifications.get("subGenre").get("name")
        ))

    return events, list(venues_dict.values()), list(attractions_dict.values())

In [8]:
raw_events = raw_data.get("_embedded", {}).get("events", [])

In [12]:
raw_events[0].get("dates").get("status").get("code")

'onsale'

In [14]:
raw_events[0].get("sales").get("public").get("startDateTime")

'2026-05-15T17:00:00Z'

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()
db_host = os.getenv("DB_HOST")
db_port = int(os.getenv("DB_PORT"))
db_user = os.getenv("DB_USER") 
db_password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")

In [29]:
def load_data(events: List[Event], venues: List[Venue], attractions: List[Attraction]):
    conn = psycopg2.connect(
        host=db_host,
        dbname=db_name,
        user=db_user,
        password=db_password,
        port=db_port
    )
    cursor = conn.cursor()

    # INSERT OR IGNORE skips rows if primary key exists (SQLite / MySQL)
    # For PostgreSQL, use: "INSERT INTO venues VALUES (...) ON CONFLICT (venue_id) DO NOTHING"
    for v in venues:
        cursor.execute("""
            INSERT INTO venues VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (venue_id) DO NOTHING;
        """, (v.venue_id, v.name, v.city, v.state, v.country, v.postal_code, v.longitude, v.latitude))

    for a in attractions:
        cursor.execute("""
            INSERT INTO attractions VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (attraction_id) DO NOTHING;
        """, (a.attraction_id, a.name, a.segment, a.genre, a.subgenre))

    for e in events:
        cursor.execute("""
            INSERT INTO events VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (event_id) DO UPDATE SET
                min_price = EXCLUDED.min_price,
                max_price = EXCLUDED.max_price,
                date = EXCLUDED.date;
        """, (e.event_id, e.name, e.date, e.min_price, e.max_price, e.segment, e.genre, e.subgenre, e.venue_id, e.attraction_id))

    conn.commit()
    conn.close()

In [30]:
def run_pipeline():
    raw_data = get_data()
    if raw_data:
        events, venues, attractions = transform_data(raw_data)
        load_data(events, venues, attractions)
        print("Pipeline finished successfully!")

In [31]:
run_pipeline()

Pipeline finished successfully!


In [20]:
for i in range(5,10):
    print(i)

5
6
7
8
9
